# 📬 Smart Email Intelligence Suite
### Week 1 · Day 1 — Commercial Use Case Homework

---

**Concept:** Most email clients only solve *one* problem at a time. This notebook demonstrates a richer commercial feature set that a premium email tool (think Superhuman, Spark, or a future Gmail AI layer) could offer — all from a single LLM call per email.

Given an email body, the model returns a structured **Email Intelligence Card** containing:

| Output | Description |
|--------|-------------|
| 🏷️ **Subject Lines** | 3 options — formal, punchy, and emoji-friendly — ranked by click-through likelihood |
| 🚨 **Priority Tag** | `[URGENT]` / `[ACTION REQUIRED]` / `[FYI]` / `[WAITING]` |
| ⚡ **One-Line Reply** | A suggested first sentence to start writing back |
| 📋 **CRM Summary** | A ≤25-word note suitable for pasting into Salesforce / HubSpot |

This pattern — **one prompt, structured multi-output** — is the foundation of real commercial LLM products.

---

## 1. Setup

In [1]:
# Install / import dependencies
# (All packages are included in the course environment — no extra installs needed)

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)  # Load OPENAI_API_KEY from your .env file

client = OpenAI()
MODEL = "gpt-4o-mini"  # Fast & cheap — perfect for a production email feature

print("✅ OpenAI client ready.")

✅ OpenAI client ready.


## 2. The System Prompt — Engineering the Intelligence Card

The key design decision: ask the model to return **structured JSON** so every downstream feature (subject line picker, CRM sync, priority filter) can be built on top without parsing free-form text.

In [2]:
SYSTEM_PROMPT = """
You are an AI assistant embedded in a premium email client.
Your job is to analyze an email body and return a structured JSON object, nothing else.

Return ONLY valid JSON in exactly this shape (no markdown fences, no extra keys):

{
  "subject_lines": {
    "formal":  "<Professional subject line, ≤10 words>",
    "punchy":  "<Short, direct, high-urgency style, ≤7 words>",
    "friendly": "<Conversational + one relevant emoji, ≤8 words>"
  },
  "priority_tag": "<Exactly one of: URGENT | ACTION REQUIRED | FYI | WAITING>",
  "priority_reason": "<One sentence explaining the tag choice>",
  "one_line_reply": "<A warm, professional opening sentence to start a reply>",
  "crm_summary": "<≤25 words capturing who wants what and by when, for a CRM note>"
}

Rules:
- Be concise. Every word must earn its place.
- Subject lines must NOT start with 'Re:' or 'Fwd:'.
- The CRM summary must be a complete sentence in past tense.
- If a deadline is mentioned, include it verbatim in the CRM summary.
""".strip()

## 3. Core Function

In [3]:
def analyze_email(email_body: str) -> dict:
    """
    Send an email body to the LLM and return a parsed Email Intelligence Card.
    
    Args:
        email_body: The raw text of the email (no subject needed).
    
    Returns:
        A dict with keys: subject_lines, priority_tag, priority_reason,
        one_line_reply, crm_summary.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Email body:\n\n{email_body}"}
        ],
        temperature=0.3,   # Low temp = consistent, reliable output
        max_tokens=400,
    )
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)  # Structured JSON → easy to wire into any UI or backend


def display_card(card: dict, email_preview: str = "") -> None:
    """Pretty-print the Email Intelligence Card in the notebook."""
    if email_preview:
        print(f"📧 EMAIL PREVIEW")
        print(f"   {email_preview[:120].strip()}...")
        print()

    s = card["subject_lines"]
    print("🏷️  SUBJECT LINE OPTIONS")
    print(f"   Formal   → {s['formal']}")
    print(f"   Punchy   → {s['punchy']}")
    print(f"   Friendly → {s['friendly']}")
    print()
    print(f"🚨 PRIORITY TAG: [{card['priority_tag']}]")
    print(f"   Why: {card['priority_reason']}")
    print()
    print(f"⚡ SUGGESTED REPLY OPENER")
    print(f"   \"{card['one_line_reply']}\"")
    print()
    print(f"📋 CRM SUMMARY")
    print(f"   {card['crm_summary']}")
    print("-" * 60)

print("✅ Functions defined.")

✅ Functions defined.


## 4. Test Emails

Three realistic scenarios that stress-test all four output types.

In [4]:
TEST_EMAILS = {

    "Sales Opportunity (Urgent)": """
Hi Sarah,

Quick one — our procurement committee met this morning and we've shortlisted your platform
alongside two competitors. We need a revised quote with enterprise pricing and a security
addendum by this Friday (April 5th) or we'll have to move forward with one of the other vendors.
Budget is confirmed at $180k annually. The decision-maker is our CTO, James Reyes.

Looking forward to hearing from you.

Best,
Michael Torres
VP of Operations, Meridian Logistics
""",

    "Internal Status Update (FYI)": """
Hey team,

Just a heads-up that the Q1 performance dashboard has been updated in Notion with final
numbers from finance. NPS is up 4 points to 67, churn held steady at 1.8%, and we hit
103% of MRR target. Great work everyone, no action needed on your end, just wanted
you all to have the latest.

– Dana
""",

    "Support Escalation (Action Required)": """
Hello,

I'm writing on behalf of TechnoFab Inc. (Account #TF-8821). We've been experiencing
critical data sync failures between your API and our ERP system since Tuesday afternoon.
Three production batches have been delayed as a result, costing us approximately $12,000
per day in downtime. Our engineering team has already opened ticket #80442 but hasn't
received a response in 48 hours.

We need an engineer on a call by tomorrow morning at the absolute latest.

Regards,
Priya Nair
Head of IT, TechnoFab Inc.
"""
}

print(f"✅ {len(TEST_EMAILS)} test emails loaded.")

✅ 3 test emails loaded.


## 5. Run the Suite

In [5]:
for label, email_body in TEST_EMAILS.items():
    print(f"\n{'='*60}")
    print(f"  SCENARIO: {label}")
    print(f"{'='*60}\n")
    
    card = analyze_email(email_body)
    display_card(card, email_preview=email_body)


  SCENARIO: Sales Opportunity (Urgent)

📧 EMAIL PREVIEW
   Hi Sarah,

Quick one — our procurement committee met this morning and we've shortlisted your platform
alongside two com...

🏷️  SUBJECT LINE OPTIONS
   Formal   → Request for Revised Quote and Addendum
   Punchy   → Quote Needed by Friday!
   Friendly → Hi Sarah, we need your help! 😊

🚨 PRIORITY TAG: [ACTION REQUIRED]
   Why: A revised quote and addendum are needed by Friday.

⚡ SUGGESTED REPLY OPENER
   "Thank you for considering our platform, Michael."

📋 CRM SUMMARY
   Michael requested a revised quote and security addendum by April 5th.
------------------------------------------------------------

  SCENARIO: Internal Status Update (FYI)

📧 EMAIL PREVIEW
   Hey team,

Just a heads-up that the Q1 performance dashboard has been updated in Notion with final
numbers from finance...

🏷️  SUBJECT LINE OPTIONS
   Formal   → Q1 Performance Dashboard Update
   Punchy   → Q1 Dashboard Updated!
   Friendly → Great news on Q1 performa

## 6. Try Your Own Email

Paste any email body below and run the cell.

In [6]:
my_email = """
Ms. Marie,

Receipt acknowledged. But yes, I have every question. The bill is not itemized for the services and the rates are not disclosed. It is merely a total sum without any explanation except 152 pages. Also, when we ordered services we advised that we want the absolute bare-bones. We have no interest in a condensed transcript. Also, we have received nothing, so I do not understand how there are shipping and handling charges. Or are there? This invoice asks more questions than it answers. Please provide us with an itemized invoice showing what you are charging us for, and what the rates are. Thank you.
"""

if my_email.strip() != "Paste your email here.":
    card = analyze_email(my_email)
    display_card(card, email_preview=my_email)
else:
    print("👆 Replace the placeholder text with a real email body and re-run!")

📧 EMAIL PREVIEW
   Ms. Marie,

Receipt acknowledged. But yes, I have every question. The bill is not itemized for the services and the rat...

🏷️  SUBJECT LINE OPTIONS
   Formal   → Request for Itemized Invoice Clarification
   Punchy   → Invoice Questions Need Answers!
   Friendly → Hi Marie, can we clarify the invoice? 😊

🚨 PRIORITY TAG: [ACTION REQUIRED]
   Why: The client needs an itemized invoice to address their concerns.

⚡ SUGGESTED REPLY OPENER
   "Thank you for your email, and I appreciate your patience."

📋 CRM SUMMARY
   The client requested an itemized invoice for clarification on charges.
------------------------------------------------------------


---
## 7. What Makes This a Real Commercial Feature?

| Concern | How this notebook addresses it |
|---------|--------------------------------|
| **Latency** | Single API call per email; `gpt-4o-mini` averages ~600ms |
| **Cost** | ~$0.0002 per email at current `gpt-4o-mini` pricing — viable at scale |
| **Reliability** | Structured JSON output + low temperature = consistent parsing |
| **Extensibility** | Add `"calendar_event"`, `"action_items"`, or `"sentiment"` keys with zero refactoring |
| **Privacy** | In production, swap `client = OpenAI()` for an on-prem Ollama endpoint — no data leaves your infrastructure |

> **Key LLM engineering insight:** Asking for *structured* output (JSON) rather than free-form prose is the bridge between a cool demo and a shippable product. The moment your downstream code can reliably `json.loads()` a response, you've unlocked every integration — CRMs, mobile push, analytics pipelines, and more.

---
*Week 1 · Day 1 Community Contribution — LLM Engineering: Master AI and Large Language Models (Udemy)*